> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [ワークフロー概要](#ワークフロー概要)
- [Sequential Workflow](#sequential-workflow)
- [Group Chat Workflow](#group-chat-workflow)
- [Human-in-loop Workflow](#human-in-loop-workflow)

## 🎯 学習目標

- Microsoft Foundryワークフローの核心概念の理解
- Sequential Workflowを通じた順次タスクフローの構築
- Group Chat Workflowを通じたマルチエージェント協調の実装
- Human-in-loopパターンを通じた人間の介入ポイントの設定
- ワークフローのデプロイとプログラマティック呼び出し

## ⏱️ 予想所要時間

約20分

## 環境設定

ワークフロー 実行を ための 設定です.

In [ ]:
# 環境変数 ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを 見つけられるように)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# が前 ノートブックで 保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境変数でも 設定 (他のツールが使用できるように)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定ファイル '{config_file}'で 環境変数を ロードしました.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイルを 見つかりません.")
    print("💡 01-setup.ipynbを 先に実行して環境を 設定してください.")
    raise

# 必須 パッケージ インストール
%pip install -q azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用するプロジェクトエンドポイント: {PROJECT_ENDPOINT}")

## Sequential Workflow用 エージェント 作成

Sequential Workflowで 使用するエージェントを作成します.
- **TravelPlannerAgent**: 旅行 目的地と 日程を基獲
- **LocalAgent**: 現地 情報を 追加 (Web Search 使用)
- **TravelSummaryAgent**: 最終 概要 および チェックリスト 作成

In [ ]:
# TravelPlannerAgent 作成
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

TRAVEL_PLANNER_INSTRUCTIONS = """あなたは 旅行 計画 専門がです.

ロール:
1. ユーザーの 旅行 要件を 分析します
2. 目的地の 主要 観光地, グルメ, 宿泊施設を おすすめします
3. 日程別 旅行 日程を 具体的で 作成します
4. 予想 コストと 準備物を 提示します

出力 形式:
- 目的地 概要
- 日程別 日程 (朝/昼/夜 活動)
- おすすめ 宿泊施設
- 予想 コスト
- 準備物 リスト

次のエージェントに 渡す 情報: 全体 旅行 計画"""

agent_travel_planner = client.agents.create_agent(
    model="gpt-5.1",
    name="TravelPlannerAgent",
    instructions=TRAVEL_PLANNER_INSTRUCTIONS
)

print(f"✅ TravelPlannerAgent 作成 完了!")
print(f"   ID: {agent_travel_planner.id}")
print(f"   Name: {agent_travel_planner.name}")

In [ ]:
# LocalAgent 作成 (Web Search も旧 使用)
LOCAL_AGENT_INSTRUCTIONS = """あなたは 現地 情報 専門がです.

ロール:
1. が前 エージェントの 旅行 計画を 受け取ります
2. Web searchを 使用して 最新 現地 情報を 検索します
3. 実時間 情報を 追加します:
   - 現在 天気 および 気候
   - 現地 フェスティバル および がベント
   - 交通 情報 (路線, 料金, 所要時間)
   - 営業時間 および 予約 情報
   - 現地 文化 および 注意事項

出力 形式:
- 元の 日程 + 現地 情報 補強
- 交通手段 詳細 情報
- 予約 必要 場所 リスト
- 現地 ヒント

次のエージェントに 渡す 情報: 現地 情報が 追加された 旅行 計画"""

agent_local = client.agents.create_agent(
    model="gpt-5.1",
    name="LocalAgent",
    instructions=LOCAL_AGENT_INSTRUCTIONS,
    tools=[{"type": "web_search"}]
)

print(f"✅ LocalAgent 作成 完了!")
print(f"   ID: {agent_local.id}")
print(f"   Tools: web_search")

In [ ]:
# TravelSummaryAgent 作成
TRAVEL_SUMMARY_INSTRUCTIONS = """あなたは 旅行 計画 整理 専門がです.

ロール:
1. が前 エージェントたちの 情報を 総合します
2. 実行 可能な 最終 計画で 整理します
3. チェックリストを 作成します

出力 形式:
📋 旅行 概要
- 目的地: 
- 期間:
- 予算:

📅 日程 概要 (一目に 見るは 日程)

✅ 出発 前 チェックリスト
- [ ] 項目1
- [ ] 項目2

🎒 準備物 チェックリスト

📞 緊急 連絡先 および 有用な 情報

最終 出力: プリント 可能な 旅行 ガイド"""

agent_travel_summary = client.agents.create_agent(
    model="gpt-5.1",
    name="TravelSummaryAgent",
    instructions=TRAVEL_SUMMARY_INSTRUCTIONS
)

print(f"✅ TravelSummaryAgent 作成 完了!")
print(f"   ID: {agent_travel_summary.id}")

## Group Chat Workflow用 エージェント 作成

Group Chat Workflowで 使用するエージェントを作成します.
- **StudentAgent**: 質問に 回答するは 学生 ロール
- **TeacherAgent**: 回答を 評価して フィードバックを 主は 教師 ロール

In [ ]:
# StudentAgent 作成
STUDENT_INSTRUCTIONS = """あなたは 問題に 答えるは エージェントよ. 質問が 来たら, 常に 回答して.

ロール:
1. ユーザーの 質問を が理解し 回答を 作成します
2. 最初の 番目 時もでは デフォルト的な 回答を 提供します
3. TeacherAgentの フィードバックを 受けて 回答を 改善します
4. すべての 要件が 充足される 時まで 回答を 修正します

回答 時 考慮事項:
- 日程 (日付, 時間)
- コスト (予算, が的)
- 好み (好みも, スタイル)
- 制約事項 (制限事項, 条件)

改善が 必要すると TeacherAgentの フィードバックを 反映して 回答を 補完します."""

agent_student = client.agents.create_agent(
    model="gpt-5.1",
    name="StudentAgent",
    instructions=STUDENT_INSTRUCTIONS
)

print(f"✅ StudentAgent 作成 完了!")
print(f"   ID: {agent_student.id}")

In [ ]:
# TeacherAgent 作成
TEACHER_INSTRUCTIONS = """あなたは 回答を 評価するは エージェントよ. 回答が 日程, コスト, 好み など 様々な 条件に に対する 考慮を したなら [COMPLETE]がと 答えて. でなければ, COMPLETEを 表示しない ではなく, 修正を リクエストして.

評価 基準:
1. 日程: 具体的な 日付, 時間, 期間が 含まれたはが?
2. コスト: 予算, が的, コスト 情報が 含まれたはが?
3. 好み: ユーザーの 好みも私 スタイルを 考慮しはが?
4. 実用性: 実際で 実行 可能な 計画のが?
5. 完成も: すべての 必要な 情報が 含まれたはが?

レスポンス 形式:
評価 完了 時: "[COMPLETE] すべての 条件が 充足されました."
改善 必要 時: "次の 事項を 補完してください: [具体的な フィードバック]"

重要: [COMPLETE]は すべての 基準が 充足されたを 時だけ 使用します."""

agent_teacher = client.agents.create_agent(
    model="gpt-5.1",
    name="TeacherAgent",
    instructions=TEACHER_INSTRUCTIONS
)

print(f"✅ TeacherAgent 作成 完了!")
print(f"   ID: {agent_teacher.id}")

In [ ]:
# 作成された エージェント リスト 確認
print("=" * 80)
print("ワークフロー用 エージェント リスト")
print("=" * 80)

agents = client.agents.list_agents()
workflow_agents = ["TravelPlannerAgent", "LocalAgent", "TravelSummaryAgent", "StudentAgent", "TeacherAgent"]

for agent in agents:
    if agent.name in workflow_agents:
        tools = "None"
        if agent.tools:
            tools = ", ".join([t.type if hasattr(t, 'type') else str(t) for t in agent.tools])
        print(f"\n📌 {agent.name}")
        print(f"   ID: {agent.id}")
        print(f"   Model: {agent.model}")
        print(f"   Tools: {tools}")

print("\n" + "=" * 80)
print("\n💡 が第 Azure Portalで ワークフローを 作成してください:")
print("   https://ai.azure.com > Build > Workflows > + Create workflow")
print("\n   Sequential Workflow:")
print("     Step 1: TravelPlannerAgent")
print("     Step 2: LocalAgent") 
print("     Step 3: TravelSummaryAgent")
print("\n   Group Chat Workflow:")
print("     Participants: StudentAgent, TeacherAgent")
print("     Termination: [COMPLETE] 含む 時")

### ワークフロー 呼び出し 例

In [ ]:
# Sequential Workflow 呼び出し 例
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ResponseStreamEventType

WORKFLOW_NAME = "Sequential-Workflow"  # ⚠️ ポータルで 作成した ワークフロー 名前
WORKFLOW_VERSION = "1"

# AI Project クライアント 作成
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

with project_client:
    workflow = {
        "name": WORKFLOW_NAME,
        "version": WORKFLOW_VERSION,
    }
    
    # OpenAI クライアントの取得
    openai_client = project_client.get_openai_client()

    # 会話 作成
    conversation = openai_client.conversations.create()
    print(f"Created conversation (id: {conversation.id})")

    # ワークフロー 呼び出し (ストリーミング)
    print(f"\nCalling workflow: {WORKFLOW_NAME}...\n")
    stream = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": workflow["name"], "type": "agent_reference"}},
        input="済州島2泊3日旅行 日程 作って",
        stream=True,
        metadata={"x-ms-debug-mode-enabled": "1"},
    )

    # ストリーミング がベント 処理
    for event in stream:
        if event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DONE:
            print("\t", event.text)
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_ADDED and event.item.type == "workflow_action":
            print(f"\n{'='*60}")
            print(f"Actor - '{event.item.action_id}':")
            print(f"{'='*60}")
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_DONE and event.item.type == "workflow_action":
            print(f"\n✓ Workflow Item '{event.item.action_id}' is '{event.item.status}'")
            print(f"  (previous item was: '{event.item.previous_action_id}')")
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DELTA:
            print(event.delta, end="", flush=True)

    # 整理
    print("\n\n✅ Workflow completed!")
    openai_client.conversations.delete(conversation_id=conversation.id)
    print("Conversation deleted")

## ワークフロー 作成

**⚠️ 重要**: ワークフローは 現在 Azure Portalでだけ 作成 可能です.

### Azure Portalで ワークフロー 作成 方法:

1. [Azure AI Foundry](https://ai.azure.com) 接続
2. **Build > Workflows** メニュー
3. **+ Create workflow** クリック
4. ワークフロー タイプ 選択:
   - Sequential Workflow
   - Group Chat Workflow  
   - Human-in-loop Workflow

### ワークフロー 構成 例:

**Sequential Workflow (旅行 計画)**
```
User Input → SearchAgent → PlannerAgent → ReviewAgent → Output
```

**Group Chat Workflow (学習 議論)**
```
User Question → StudentAgent ↔ TeacherAgent → Consensus
```

作成が 完了なると 以下 コードで 実行する できる あります.

## ワークフロー 実行

ポータルで 作成した ワークフローを Python コードで 実行します.

In [ ]:
# ワークフロー 実行 コード
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ResponseStreamEventType

# クライアント 作成
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# ⚠️ ワークフロー 名前と バージョン 設定 (ポータルで 作成した ことで 変更)
WORKFLOW_NAME = "Sequential-Workflow"  # ⚠️ 変更 必要
WORKFLOW_VERSION = "1"

with project_client:
    # OpenAI クライアントの取得
    openai_client = project_client.get_openai_client()
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    print(f"✅ Conversation 作成: {conversation.id}")
    
    # ワークフロー 実行 (ストリーミング)
    print(f"\n🚀 ワークフロー 実行 中: {WORKFLOW_NAME}...\n")
    print("=" * 80)
    
    stream = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": WORKFLOW_NAME, "type": "agent_reference"}},
        input="済州島2泊3日旅行 日程 作って",  # ⚠️ 望むは 質問で 変更
        stream=True,
        metadata={"x-ms-debug-mode-enabled": "1"}
    )
    
    # ストリーミング 結果 処理
    for event in stream:
        if event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DELTA:
            print(event.delta, end="", flush=True)
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_ADDED and event.item.type == "workflow_action":
            print(f"\n\n🤖 Actor: {event.item.action_id}")
        elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_DONE and event.item.type == "workflow_action":
            print(f"\n✅ '{event.item.action_id}' 完了 (状態: {event.item.status})")
    
    print("\n" + "=" * 80)
    print("\n✅ ワークフロー 実行 完了!")
    
    # Conversation 削除
    openai_client.conversations.delete(conversation_id=conversation.id)
    print(f"🗑️ Conversation 削除になる")

### ワークフロー 設計

**ポータルで 構成:**

```
TravelPlannerAgent → [ユーザー 承認] → LocalAgent → TravelSummaryAgent
```

**Approval 設定:**
- Approval message: "作成された 旅行 計画を レビューしてください. 承認しますか?"
- Options: Approve / Reject / Modify
- Timeout: 24時間

### 💡 Human-in-loop ベスト 事例

**推奨事項:**
- 承認 ポイントを 明確に 表示
- タイムアウト 設定で 無限 待機 防止
- ユーザーに コンテキスト 提供 (が前 会話 概要)
- 簡単な 承認 オプション 提供 (はい/いいえ/修正)

**避けるべき する こと:**
- あまりにも 多いは 承認 ポイント
- 不明確な 承認 質問
- 長い タイムアウト (ユーザー 経験 低下)
- 承認 後 元に戻す 不可能な 構造

## 📚 追加リソース

- [Microsoft Foundry Workflows 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/workflow?view=foundry)
- [Microsoft Agent Framework Workflows Orchestrations パターン](https://learn.microsoft.com/en-us/agent-framework/user-guide/workflows/orchestrations/overview)

## 次のステップ

複雑な ワークフローを 構築しました! が第 エージェントと ワークフローの パフォーマンスを 評価する方法を 学習します:

➡️ **[06. 評価](./06-evaluations.ipynb)**: エージェント および ワークフローの 品質を 体系的で 評価します.